# PD02 · Consultar y resumir EVA

[Índice y distribución de las cuatro horas](../ejercicios/README.md) · [Guía de pandas](../docs/manual-pandas.md)

**Tiempo de trabajo: 70 minutos.** El objetivo es traducir una pregunta a filtros verificables y resúmenes. La pregunta de trabajo es: ¿cómo se distribuyen los registros agrícolas de Boyacá entre cultivos durante 2023 y 2024?

| Símbolo | Uso |
|:---:|---|
| 📘 | Concepto o función de apoyo. |
| ✅ | Evidencia que debe quedar visible. |
| ⚠️ | Límite de interpretación. |

Esta plantilla se encuentra en `ejercicios/PD02_consultas_eva.ipynb`. Antes de comenzar, se crea una copia en `soluciones/PD02_consultas_eva.ipynb` dentro del repositorio de cada estudiante. El desarrollo y la entrega se realizan en esa copia; la plantilla de `ejercicios/` se conserva como enunciado. En JupyterLab se puede duplicar el archivo y mover la copia a `soluciones/`, manteniendo el nombre indicado; si ya existe una solución, se continúa sobre ella. Las instrucciones se presentan en celdas Markdown; las celdas de código marcadas como **Desarrollo** contienen el espacio de trabajo. Cada actividad incluye una celda para explicar el resultado.

Se utiliza el kernel del entorno `.venv` del curso. Los CSV ya deben estar descargados mediante `kit/00_datos.py`. La celda inicial prepara las rutas y carga únicamente las entradas de este ejercicio; el desarrollo del análisis corresponde al estudiante. Este notebook se ejecuta de forma independiente.

**Apoyo en el manual:** secciones 4, 5, 8, 10 y las recetas de porcentajes y equivalencia de la sección 11.

## Ubicación de trabajo: la raíz del repositorio

📘 **La raíz del repositorio** es la carpeta principal de la copia del proyecto. En la instalación del curso se llama `big-data-postgraduate` y contiene `README.md`, `kit/`, `ejercicios/` y `soluciones/`:

```text
bigdata/
└── big-data-postgraduate/       ← raíz del repositorio
    ├── README.md
    ├── kit/
    ├── ejercicios/              ← enunciados y plantillas
    └── soluciones/              ← notebooks resueltos
```

**Desde la terminal de Ubuntu en WSL.** `pwd` muestra la carpeta actual, `cd` cambia de carpeta y `ls` muestra su contenido. Los siguientes comandos se ejecutan en la terminal, no en una celda Python:

```bash
pwd
cd /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
pwd
ls
```

✅ Después de `cd`, el segundo `pwd` debe mostrar `/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate`. En la salida de `ls` deben aparecer `README.md`, `kit`, `ejercicios` y `soluciones`. La carpeta `bigdata` es la carpeta contenedora; la raíz del proyecto es su subcarpeta `big-data-postgraduate`.

La ruta anterior corresponde a la configuración de la guía WSL. Si el repositorio se clonó en otra ubicación, se sustituye por la ruta real de esa copia. Si `cd` informa que la carpeta no existe, se revisa esa ubicación antes de continuar.

**Desde una subcarpeta conocida.** `..` significa «carpeta superior». Se comprueba primero la ubicación con `pwd` y se utiliza la alternativa correspondiente:

| Ubicación actual, relativa al repositorio | Comando para volver a la raíz |
|---|---|
| `ejercicios/`, `soluciones/` o `kit/` | `cd ..` |
| `kit/Notebooks/` | `cd ../..` |

Al llegar a la raíz, se puede activar el entorno y abrir JupyterLab desde la terminal:

```bash
source .venv/bin/activate
jupyter lab
```

**Dentro de un notebook ya abierto.** La terminal y el kernel de Python mantienen sus propios directorios de trabajo. La celda de carga siguiente localiza la raíz y la guarda en `REPO`, aunque el notebook esté abierto desde `soluciones/` o `ejercicios/`. Después de ejecutar esa celda, el siguiente bloque puede ejecutarse en una celda Python para comprobar la ubicación y cambiar el directorio del kernel a la raíz:

```python
import os
from pathlib import Path

print("Directorio antes del cambio:", Path.cwd())
print("Raíz detectada:", REPO)
os.chdir(REPO)
print("Directorio después del cambio:", Path.cwd())
```

✅ `Path.cwd()` debe coincidir con `REPO` después del cambio. Esta operación cambia la base para interpretar rutas relativas; el archivo del notebook permanece en `soluciones/`. La carga de datos de estos ejercicios también funciona sin ese cambio porque construye las rutas a partir de `REPO`.

## Preparación

Se ejecuta la siguiente celda y se comprueba que el intérprete pertenece al entorno del curso y que pandas corresponde a la versión 2.2.3. Las rutas se calculan desde el repositorio, sin introducir rutas personales de Windows, WSL o macOS.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

actual = Path.cwd().resolve()
REPO = next(
    (ruta for ruta in [actual, *actual.parents]
     if (ruta / "kit/fuentes.json").is_file()
     and (ruta / "kit/00_datos.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("El notebook debe abrirse dentro del repositorio del curso")

RAW = REPO / "kit/data/raw"
OUT = REPO / "kit/salidas/soluciones/PD02"
archivos = ['eva.csv']
faltantes = [nombre for nombre in archivos if not (RAW / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan CSV en {RAW}: {faltantes}")
OUT.mkdir(parents=True, exist_ok=True)

opciones = dict(sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
eva = pd.read_csv(RAW / "eva.csv", **opciones)


print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)
print("pandas:", pd.__version__)
print("Repositorio:", REPO)
print("Salidas:", OUT)

Python: 3.12.15
Intérprete: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/.venv/bin/python
pandas: 2.2.3
Repositorio: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
Salidas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD02


## 1. Preparación de columnas · 10 minutos

Inspeccionar la tabla y crear `codigo_departamento` a partir de `Código Dane departamento`, `anio` desde `Año`, `area_ha` desde `Área cosechada` y `produccion_t` desde `Producción`. El código se conserva como texto; año, área y producción se convierten a tipos numéricos. Registrar las ausencias resultantes.

📘 **Apoyo:** `.str.strip`, `pd.to_numeric`, `astype("Int64")` e `isna`.

✅ **Comprobación:** Las columnas originales se conservan y no se eliminan filas. El corte de entrada contiene 166.732 registros.

In [2]:
from IPython.display import Markdown
import numpy as np
fmt = lambda n: f"{n:,}".replace(",", ".")
total_entrada = len(eva)

eva["codigo_departamento"] = eva["Código Dane departamento"].str.strip()
eva["anio"] = pd.to_numeric(eva["Año"].str.strip(), errors="coerce").astype("Int64")
eva["area_ha"] = pd.to_numeric(eva["Área cosechada"].str.strip(), errors="coerce")
eva["produccion_t"] = pd.to_numeric(eva["Producción"].str.strip(), errors="coerce")

pares = [("Año", "anio"), ("Área cosechada", "area_ha"), ("Producción", "produccion_t")]
filas = []
for original, nueva in pares:
    vacio = eva[original].str.strip().eq("")
    ausente = eva[nueva].isna()
    filas.append({
        "columna": nueva,
        "tipo": str(eva[nueva].dtype),
        "vacios": int(vacio.sum()),
        "fallos_conversion": int((ausente & ~vacio).sum()),
        "ausentes_total": int(ausente.sum()),
    })
revision_tipos = pd.DataFrame(filas)
display(revision_tipos)
display(eva[["codigo_departamento", "anio", "area_ha", "produccion_t"]].dtypes.to_frame("tipo"))

print("Ejemplos de áreas que no se pudieron convertir:")
display(eva.loc[eva["area_ha"].isna() & eva["Área cosechada"].str.strip().ne(""), ["Área cosechada"]].head(5))

assert len(eva) == total_entrada
r = revision_tipos.set_index("columna")


,columna,tipo,vacios,fallos_conversion,ausentes_total
0,anio,Int64,0,0,0
1,area_ha,float64,0,0,0
2,produccion_t,float64,0,0,0


,tipo
codigo_departamento,object
anio,Int64
area_ha,float64
produccion_t,float64


Ejemplos de áreas que no se pudieron convertir:


,Área cosechada


**Interpretación del resultado:**
Se crearon cuatro columnas de trabajo sin tocar las originales y sin eliminar filas: la tabla sigue con 166.732 registros (el corte esperado es 166.732). El código del departamento se dejó como texto porque es un identificador y no se suma ni se promedia; año, área y producción sí se pasaron a número. Las ausencias resultantes son: año 0 (0 vacíos y 0 fallos), área 0 (0 vacíos y 0 fallos) y producción 0 (0 vacíos y 0 fallos). Estos valores quedan como NaN y no se rellenaron con cero: una producción de cero sería un dato inventado, y además más adelante esas filas simplemente no pasan los filtros de área positiva y producción no negativa.


## 2. Selección con máscara y loc · 15 minutos

Seleccionar el departamento `15` (Boyacá), los años 2023 y 2024, área cosechada positiva y producción no negativa. Guardar todas las columnas de las filas seleccionadas en `consulta_loc`, utilizando `.copy()`. Presentar el total original, el seleccionado y la diferencia.

📘 **Apoyo:** `.eq`, `.isin`, `.gt`, `.ge`, `&` y `.loc`.

✅ **Comprobación:** Todas las filas seleccionadas cumplen simultáneamente las cuatro condiciones. La diferencia respecto a la entrada incluye registros fuera del alcance territorial o temporal; no implica que todos sean errores.

In [3]:
COD = "15"   # Boyacá

mascara = (
    eva["codigo_departamento"].eq(COD)
    & eva["anio"].isin([2023, 2024])
    & eva["area_ha"].gt(0)
    & eva["produccion_t"].ge(0)
)
consulta_loc = eva.loc[mascara].copy()

# Todas las filas cumplen las cuatro condiciones a la vez
assert consulta_loc["codigo_departamento"].eq(COD).all()
assert consulta_loc["anio"].isin([2023, 2024]).all()
assert consulta_loc["area_ha"].gt(0).all()
assert consulta_loc["produccion_t"].ge(0).all()

conteos = pd.DataFrame({
    "concepto": ["total original", "seleccionado", "diferencia"],
    "filas": [total_entrada, len(consulta_loc), total_entrada - len(consulta_loc)],
})
display(conteos)

fuera_depto = int(eva["codigo_departamento"].ne(COD).sum())
sel, dif = len(consulta_loc), total_entrada - len(consulta_loc)


,concepto,filas
0,total original,166732
1,seleccionado,4924
2,diferencia,161808


**Interpretación del resultado:**

La consulta deja 4.924 filas de 166.732, es decir 3.0 %, y quedan por fuera 161.808. Combiné las cuatro condiciones con & (departamento 15, años 2023 y 2024, área mayor que cero y producción mayor o igual a cero) y guardé todas las columnas con .copy() para trabajar sobre una tabla independiente y evitar advertencias de pandas. De las filas excluidas, 149.053 pertenecen a otros departamentos; el resto queda por año distinto de 2023-2024 o por área o producción ausente o inválida. Esa diferencia no significa que todo sea un error: la mayoría es simplemente información que está fuera del alcance de la pregunta.

## 3. Consulta equivalente con query · 10 minutos

Repetir exactamente la selección anterior mediante `.query`, con `engine="python"`, y guardar el resultado en `consulta_query`. Usar una variable para el código departamental y referenciarla con `@`. Comparar ambos DataFrames sin eliminar columnas.

📘 **Apoyo:** `.query`, `@variable` y `pd.testing.assert_frame_equal`.

✅ **Comprobación:** La comparación debe confirmar las mismas filas, columnas y valores. Si falla, se revisan las condiciones y los tipos antes de continuar.

In [4]:
cod = "15"
consulta_query = eva.query(
    "codigo_departamento == @cod and anio in [2023, 2024] and area_ha > 0 and produccion_t >= 0",
    engine="python",
)

pd.testing.assert_frame_equal(consulta_loc, consulta_query)
print("Equivalentes:", consulta_loc.shape == consulta_query.shape, consulta_loc.shape)



Equivalentes: True (4924, 22)


**Interpretación del resultado:**

Con .query obtuve 4.924 filas y assert_frame_equal confirmó que consulta_loc y consulta_query tienen las mismas filas, columnas y valores, sin dar error. Las dos formas hacen lo mismo: .loc con máscara es más explícito y fácil de depurar condición por condición, mientras que .query se lee más parecido a una frase y permite usar la variable cod con @. Usé engine="python" como pide el taller. Para este ejercicio me quedo con .loc como base porque se ve mejor cada filtro, pero es bueno saber que ambas dan el mismo resultado.

## 4. Conteos y porcentajes · 20 minutos

Sobre `consulta_loc`, construir `resumen` con una fila por año y cultivo y una columna `registros`. Añadir el total de registros de cada año y `porcentaje_anual`. Ordenar los resultados por año y cantidad de registros descendente. Conservar los grupos con claves ausentes mediante `dropna=False`.

📘 **Apoyo:** `groupby`, `size`, `reset_index`, `transform("sum")` y `sort_values`.

✅ **Comprobación:** La suma de registros coincide con el tamaño de consulta_loc. Los porcentajes suman aproximadamente 100 % por año; se admite una pequeña tolerancia numérica y se redondea solo para mostrar.

In [5]:
col_cultivo = "Cultivo"   # si el CSV usa otro nombre, cambiarlo aquí

resumen = (
    consulta_loc.groupby(["anio", col_cultivo], dropna=False)
    .size()
    .reset_index(name="registros")
)
resumen["total_anio"] = resumen.groupby("anio", dropna=False)["registros"].transform("sum")
resumen["porcentaje_anual"] = resumen["registros"] / resumen["total_anio"] * 100
resumen = resumen.sort_values(["anio", "registros"], ascending=[True, False]).reset_index(drop=True)

assert resumen["registros"].sum() == len(consulta_loc)
suma_pct = resumen.groupby("anio")["porcentaje_anual"].sum()
assert np.allclose(suma_pct, 100)
print("Suma de porcentajes por año:")
display(suma_pct.round(4).to_frame("suma_%"))

print("Cinco cultivos con más registros por año:")
display(resumen.groupby("anio").head(5).round({"porcentaje_anual": 2}))

top = resumen.groupby("anio").head(1).set_index("anio")
frases = [f"En {a} el cultivo con más registros fue {f[col_cultivo]} con {fmt(int(f['registros']))} ({f['porcentaje_anual']:.1f} %)" for a, f in top.iterrows()]


Suma de porcentajes por año:


,suma_%
anio,
2023,100.0
2024,100.0


Cinco cultivos con más registros por año:


,anio,Cultivo,registros,total_anio,porcentaje_anual
0,2023,Papa,239,2436,9.81
1,2023,Maíz,194,2436,7.96
2,2023,Arveja,176,2436,7.22
3,2023,Frijol,162,2436,6.65
4,2023,Tomate,113,2436,4.64
96,2024,Papa,237,2488,9.53
97,2024,Maíz,196,2488,7.88
98,2024,Frijol,169,2488,6.79
99,2024,Arveja,168,2488,6.75
100,2024,Tomate,120,2488,4.82


**Interpretación del resultado:**

El resumen tiene 192 combinaciones de año y cultivo, y sus registros suman 4.924, igual que el tamaño de consulta_loc. Los porcentajes suman 100 % en cada año (se comprobó con tolerancia numérica y se redondeó solo para mostrar). En 2023 el cultivo con más registros fue Papa con 239 (9.8 %); En 2024 el cultivo con más registros fue Papa con 237 (9.5 %). Usé dropna=False para no perder grupos con clave ausente, y ordené por año y por cantidad descendente para que los cultivos más frecuentes aparezcan primero. Es importante recordar que estos porcentajes son de registros y no de área cultivada.

## 5. Tabla dinámica y conclusiones · 15 minutos

Construir una tabla dinámica a partir de `resumen`, con cultivos en filas, años en columnas y conteos como valores. Completar con cero las combinaciones sin registros. Exportar `resumen` a `OUT / "resumen_eva.csv"`, sin índice, y redactar dos conclusiones.

📘 **Apoyo:** `pivot_table`, `fill_value=0` y `to_csv`.

✅ **Comprobación:** Los totales de la tabla dinámica concilian con resumen. Las conclusiones se refieren a la cantidad de registros y no a la participación de cada cultivo en la superficie agrícola.

In [6]:
tabla = resumen.pivot_table(
    index=col_cultivo, columns="anio", values="registros", aggfunc="sum", fill_value=0
)

# Conciliación con el resumen
assert int(tabla.to_numpy().sum()) == int(resumen["registros"].sum())
assert (tabla.sum() == resumen.groupby("anio")["registros"].sum()).all()

tabla_ver = tabla.assign(Total=tabla.sum(axis=1)).sort_values("Total", ascending=False)
display(tabla_ver)

ruta = OUT / "resumen_eva.csv"
resumen.to_csv(ruta, index=False)
print("Archivo generado:", ruta.is_file(), ruta)

ceros = int((tabla == 0).sum().sum())
lider = tabla_ver.index[0]
cols_anio = [c for c in tabla_ver.columns if c != "Total"]
display(Markdown(f"""**Conclusión 1.** En Boyacá, entre 2023 y 2024, el cultivo con más registros es {lider}, con {fmt(int(tabla_ver.iloc[0]['Total']))} de {fmt(int(tabla_ver['Total'].sum()))} ({100*tabla_ver.iloc[0]['Total']/tabla_ver['Total'].sum():.1f} %). Esto describe cuántas observaciones aparecen en la fuente EVA, no cuánta superficie ocupa cada cultivo.

**Conclusión 2.** La tabla dinámica tiene {fmt(len(tabla))} cultivos y {len(cols_anio)} años, y {fmt(ceros)} combinaciones con cero. Un cero quiere decir que no se encontraron filas para esa combinación después de los filtros, no que el cultivo no exista ese año. Los totales de la tabla concilian con `resumen` ({fmt(int(resumen['registros'].sum()))} registros). Una fila EVA es una observación de la fuente (un cultivo, un lugar y un periodo), no un predio ni una persona, así que no se puede interpretar como número de agricultores ni como participación en el área agrícola."""))

anio,2023,2024,Total
Cultivo,,,
Papa,239,237,476
Maíz,194,196,390
Arveja,176,168,344
Frijol,162,169,331
Tomate,113,120,233
...,...,...,...
Dátil,1,1,2
"Otros cultivos para condimentos, bebidas medicinales y aromáticas",1,1,2
Macadamia,1,1,2


Archivo generado: True /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD02/resumen_eva.csv


**Conclusión 1.** En Boyacá, entre 2023 y 2024, el cultivo con más registros es Papa, con 476 de 4.924 (9.7 %). Esto describe cuántas observaciones aparecen en la fuente EVA, no cuánta superficie ocupa cada cultivo.

**Conclusión 2.** La tabla dinámica tiene 97 cultivos y 2 años, y 2 combinaciones con cero. Un cero quiere decir que no se encontraron filas para esa combinación después de los filtros, no que el cultivo no exista ese año. Los totales de la tabla concilian con `resumen` (4.924 registros). Una fila EVA es una observación de la fuente (un cultivo, un lugar y un periodo), no un predio ni una persona, así que no se puede interpretar como número de agricultores ni como participación en el área agrícola.

**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

## Entrega

El notebook completado se conserva como `soluciones/PD02_consultas_eva.ipynb`. La salida regenerable es `kit/salidas/soluciones/PD02/resumen_eva.csv`.

✅ Deben quedar visibles la equivalencia de consultas, la conciliación de los conteos, los porcentajes por año, la tabla dinámica y dos conclusiones.

⚠️ Una fila EVA es una observación de la fuente, no un predio ni una persona. Un porcentaje de registros no equivale a un porcentaje de área cultivada. El cero de una tabla de conteos indica que no se encontraron filas para esa combinación.